# Analisi dei risultati — confronto tra modelli

Valuta i verdetti prodotti dagli agenti contro la ground truth, con la stessa
procedura per ogni modello presente in questa cartella.

## Cosa si può misurare, e cosa no

**Il dataset contiene solo istanze positive**: ogni commit corregge una CVE reale, e
il repository è montato al commit padre, quindi la vulnerabilità c'è sempre.
Non esistono negativi.

Conseguenza diretta: si possono calcolare **recall / sensibilità** e le metriche di
classificazione CWE, ma **non** accuratezza, precisione di detection, tasso di falsi
positivi, specificità o ROC-AUC. Qualunque affermazione su 'falsi positivi' del
rilevamento richiederebbe un gruppo di controllo di commit non di sicurezza, che in
questo impianto non esiste.

## Struttura

- **Stadio 1 — copertura / detection rate.** Su tutte le istanze, in quale frazione
  l'agente riconosce che c'è qualcosa invece di rispondere 'non vulnerabile'.
- **Stadio 2 — accuratezza CWE condizionata.** Ristretto alle istanze in cui
  l'agente ha prodotto almeno una CWE, quanto è corretta rispetto alla ground truth.

La corrispondenza per *famiglia* (antenato CWE-1000) non è ancora implementata: vedi
`Literature/TODO_ANALISI_CWE.md`.


## Configurazione

`PARSE_MODE` decide come trattare i verdetti che il parser ufficiale rifiuta.
Le opzioni sono discusse nella cella successiva.


In [ ]:
import csv, json, re, statistics, collections
from pathlib import Path

BASE = Path.cwd()                      # .../agentAnalysisDocker/outputs
GROUND_TRUTH = BASE.parent / 'ground_truth.csv'
AGENTS = ['agent1', 'agent2', 'agent3']

# Cartelle modello da confrontare: tutte quelle con un log.jsonl non vuoto.
MODELS = sorted(p.name for p in BASE.iterdir()
                if (p / 'log.jsonl').exists() and (p / 'log.jsonl').stat().st_size > 0)

# 'strict'   -> solo i record che il parser ufficiale accetta (status == 'ok')
# 'tolerant' -> anche i verdetti espliciti ma con sintassi non conforme
# 'annotated'-> tolerant + annotazioni manuali da manual_verdicts.csv (se presente)
PARSE_MODE = 'tolerant'

# File opzionale per la modalita' 'annotated':
#   commit,agent,found,cwe_ids      es.  a1b2c3...,agent3,yes,"CWE-787,CWE-125"
MANUAL_CSV = BASE / 'manual_verdicts.csv'

print('modelli trovati:', MODELS)
print('ground truth   :', GROUND_TRUTH)
print('modalita\'      :', PARSE_MODE)


## Il problema delle risposte mal formattate

Alcuni run terminano l'analisi ma non emettono il blocco nel formato richiesto. Il
parser ufficiale (`verdict.py`) li marca come errore, e vengono persi verdetti
perfettamente leggibili. Il fenomeno è fortemente asimmetrico tra i modelli, quindi
la scelta di come trattarli **cambia il confronto**.

| Modalità | Cosa fa | Pro | Contro |
|---|---|---|---|
| `strict` | solo `status == 'ok'` | incontestabile, riproducibile | scarta molti run di un solo modello: confronto sbilanciato |
| `tolerant` | riconosce le varianti sintattiche osservate | deterministico e verificabile | il parser è tarato sui dati osservati |
| `annotated` | aggiunge le conclusioni espresse solo in prosa | copertura massima | soggettivo: il file di annotazioni va allegato |

Due strade scartate: un **LLM come estrattore** (non deterministico, aggiunge un
modello alla catena di misura e va comunque validato a mano) e il **rilancio con un
prompt più rigido** (cambierebbe l'esperimento, non è applicabile a posteriori).

**Politica di presentazione consigliata:** riportare sempre `strict` e `tolerant`
affiancate. Se divergono molto, la differenza è essa stessa un risultato sul
rispetto del formato.


In [ ]:
# --- Parser dei verdetti ---------------------------------------------------

def parse_strict(text):
    """Stessa logica di verdict.py: FIELD: value, tollerante solo al grassetto."""
    m = re.search(r'\**VULNERABILITY_FOUND\**\s*:\s*\**\s*([^\n*]+)', text or '', re.I)
    if not m:
        return None
    return m.group(1).strip().lower().startswith('yes')


def parse_tolerant(text):
    """Riconosce anche le varianti sintattiche osservate nei log."""
    t = text or ''
    if re.search(r'NO[_ ]VULNERABILITY[_ ]FOUND', t, re.I):
        return False
    m = re.search(r'["\']?vulnerability_found["\']?\s*[:=]\s*["\']?(true|false|yes|no)', t, re.I)
    if m:
        return m.group(1).lower() in ('true', 'yes')
    m = re.search(r'VULNERABILITY[_ ]FOUND\s*:?\s*\**\s*(yes|no|true|false)\b', t, re.I)
    if m:
        return m.group(1).lower() in ('yes', 'true')
    # blocco con l'etichetta nuda seguita da un CWE_ID valorizzato
    if re.search(r'VULNERABILITY[_ ]FOUND', t, re.I) and \
       re.search(r'CWE[_ ]ID\s*:?\s*\**\s*CWE-\d+', t, re.I):
        return True
    if re.search(r'["\']verdict["\']\s*:\s*["\']no vulnerability found', t, re.I):
        return False
    return None


CWE_RE = re.compile(r'CWE[-\s]?(\d+)', re.I)

def extract_cwes(text, window=1800):
    """CWE dichiarate nel blocco finale, **in ordine**: il prompt chiede di elencarle
    dalla piu' specifica, quindi la prima e' la risposta primaria (top-1)."""
    tail = (text or '')[-window:]
    m = re.search(r'CWE[_ ]ID\s*:?\s*\**\s*([^\n]*)', tail, re.I)
    seg = m.group(1) if m else tail
    seen, out = set(), []
    for n in CWE_RE.findall(seg):
        cid = 'CWE-' + n
        if cid not in seen:
            seen.add(cid)
            out.append(cid)
    return out


In [ ]:
# --- Ground truth ----------------------------------------------------------
# Un commit puo' correggere piu' CVE: la verita' e' l'unione delle loro CWE.
# I segnaposto NVD-CWE-* (noinfo / Other) non sono debolezze e vanno esclusi.

truth = collections.defaultdict(set)
repo_of = {}
with open(GROUND_TRUTH, newline='') as f:
    for r in csv.DictReader(f):
        repo_of[r['commit']] = r['repo_name']
        for c in (r['cwe_ids'] or '').split(','):
            c = c.strip().upper()
            if c.startswith('CWE-'):
                truth[r['commit']].add(c)

COMMITS = sorted(repo_of)
print(f'commit nella ground truth : {len(COMMITS)}')
print(f'commit con almeno una CWE : {len(truth)}')
print(f'CWE distinte              : {len({c for s in truth.values() for c in s})}')
print(f'commit con piu di una CWE : {sum(1 for s in truth.values() if len(s) > 1)}')


In [ ]:
# --- Caricamento dei risultati --------------------------------------------

def load_manual():
    """Annotazioni manuali opzionali per la modalita' 'annotated'."""
    ann = {}
    if MANUAL_CSV.exists():
        with open(MANUAL_CSV, newline='') as f:
            for r in csv.DictReader(f):
                cwes = [c.strip().upper() for c in (r.get('cwe_ids') or '').split(',') if c.strip()]
                ann[(r['commit'], r['agent'])] = (r['found'].strip().lower() in ('yes', 'true'), cwes)
    return ann

MANUAL = load_manual()


def build(model, mode):
    """Una riga per tripla (commit, agent). Se una tripla ha piu' tentativi si
    tiene il primo che produce un verdetto sotto la modalita' scelta."""
    recs = collections.defaultdict(list)
    with open(BASE / model / 'log.jsonl') as f:
        for line in f:
            r = json.loads(line)
            recs[(r['commit'], r['agent'])].append(r)

    parser = parse_strict if mode == 'strict' else parse_tolerant
    rows = []
    for commit in COMMITS:
        for agent in AGENTS:
            attempts = recs.get((commit, agent), [])
            found, cwes, src = None, [], 'nessun record'
            for r in sorted(attempts, key=lambda x: x['status'] != 'ok'):
                resp = r.get('response') or ''
                if mode == 'strict' and r['status'] != 'ok':
                    src = 'scartato (status != ok)'
                    continue
                v = parser(resp)
                if v is not None:
                    found, cwes, src = v, extract_cwes(resp), 'parser'
                    break
                src = 'verdetto non leggibile'
            if found is None and mode == 'annotated' and (commit, agent) in MANUAL:
                found, cwes = MANUAL[(commit, agent)]
                src = 'annotazione manuale'
            rows.append({'model': model, 'commit': commit, 'repo': repo_of[commit],
                         'agent': agent, 'found': found, 'pred': cwes,
                         'truth': sorted(truth.get(commit, set())), 'source': src,
                         'n_attempts': len(attempts)})
    return rows


DATA = {m: {mode: build(m, mode) for mode in ('strict', 'tolerant', 'annotated')}
        for m in MODELS}
for m in MODELS:
    n = sum(1 for r in DATA[m][PARSE_MODE] if r['found'] is not None)
    print(f'{m:26s} triple: {len(DATA[m][PARSE_MODE]):3d}  con verdetto ({PARSE_MODE}): {n:3d}')


## Stadio 1 — Copertura e detection rate

Tutte le istanze sono vulnerabili, quindi rispondere *sì* è sempre corretto e
rispondere *no* è sempre un **falso negativo**. L'unica metrica di rilevamento
onesta è dunque il **recall**, riportato in due varianti:

- **recall stretto** — sul totale delle istanze; un verdetto illeggibile conta come
  mancato rilevamento (misura il sistema nel suo insieme, formato incluso);
- **recall condizionato** — solo sulle istanze in cui l'agente ha concluso (misura
  la capacità di analisi, al netto del formato).

Il confronto tra i tre scenari è già un risultato: se la copertura cala da agent1 ad
agent3, togliere il segnale della patch riduce la capacità dell'agente di accorgersi.


In [ ]:
def stage1(rows):
    out = {}
    for agent in AGENTS:
        rs = [r for r in rows if r['agent'] == agent]
        n = len(rs)
        concluded = [r for r in rs if r['found'] is not None]
        yes = [r for r in concluded if r['found']]
        no = [r for r in concluded if not r['found']]
        out[agent] = {
            'n': n,
            'concluso': len(concluded),
            'si': len(yes),
            'no (falsi negativi)': len(no),
            'senza verdetto': n - len(concluded),
            'recall stretto': len(yes) / n if n else float('nan'),
            'recall condizionato': len(yes) / len(concluded) if concluded else float('nan'),
            'tasso di conclusione': len(concluded) / n if n else float('nan'),
        }
    return out


def show_stage1(mode):
    print(f'=== STADIO 1 — copertura  (modalita\': {mode}) ===')
    hdr = ['n', 'concluso', 'si', 'no (falsi negativi)', 'senza verdetto',
           'tasso di conclusione', 'recall condizionato', 'recall stretto']
    for model in MODELS:
        st = stage1(DATA[model][mode])
        print(f'\n{model}')
        print('  ' + 'agente'.ljust(9) + ''.join(h.rjust(22) for h in hdr))
        for agent in AGENTS:
            s = st[agent]
            cells = []
            for h in hdr:
                v = s[h]
                cells.append((f'{v:.1%}' if isinstance(v, float) else str(v)).rjust(22))
            print('  ' + agent.ljust(9) + ''.join(cells))

show_stage1(PARSE_MODE)


In [ ]:
# Confronto tra modalita' di parsing: quanto pesa il formato sul risultato.
print('=== impatto della modalita\' di parsing sul recall stretto ===')
print('  ' + 'modello'.ljust(26) + 'agente'.ljust(9) +
      'strict'.rjust(10) + 'tolerant'.rjust(10) + 'annotated'.rjust(11))
for model in MODELS:
    for agent in AGENTS:
        vals = [stage1(DATA[model][m])[agent]['recall stretto']
                for m in ('strict', 'tolerant', 'annotated')]
        print('  ' + model.ljust(26) + agent.ljust(9) +
              ''.join(f'{v:.1%}'.rjust(10 if i < 2 else 11) for i, v in enumerate(vals)))


## Stadio 2 — Accuratezza CWE condizionata

Calcolata **solo** sulle istanze in cui l'agente ha risposto *sì* e ha prodotto
almeno una CWE, e per cui la ground truth ha almeno una CWE.

Il compito è **multi-etichetta**: sia la verità sia la predizione possono contenere
più CWE. Le metriche seguono la pratica standard della classificazione multi-label:

| Metrica | Significato |
|---|---|
| **hit (any-overlap)** | almeno una CWE predetta è corretta — la più permissiva |
| **top-1** | la *prima* CWE elencata è corretta; il prompt chiede l'ordine dalla più specifica, quindi è la risposta primaria |
| **exact set** | l'insieme predetto coincide esattamente con quello vero — la più severa |
| **P/R/F1 per esempio** | precisione, recall e F1 calcolati su ogni istanza e poi mediati |
| **P/R/F1 micro** | tutte le decisioni (istanza, etichetta) messe in un'unica pentola |
| **Jaccard medio** | intersezione su unione, mediata sulle istanze |
| **F1 macro** | media per CWE — **inaffidabile qui**, vedi avvertenza |

Termine di paragone dalla letteratura: nello user study di CoLeFunDa cinque esperti
umani classificavano correttamente il **37,5%** delle CVE al primo suggerimento e il
**62,5%** entro i primi due — partendo però dalla descrizione testuale della CVE, non
dal solo codice.


In [ ]:
def stage2(rows):
    out = {}
    for agent in AGENTS:
        rs = [r for r in rows
              if r['agent'] == agent and r['found'] and r['pred'] and r['truth']]
        n = len(rs)
        if not n:
            out[agent] = {'n valutabili': 0}
            continue
        hit = top1 = exact = 0
        precs, recs_, f1s, jacs = [], [], [], []
        tp_tot = fp_tot = fn_tot = 0
        for r in rs:
            P, T = set(r['pred']), set(r['truth'])
            inter = P & T
            hit += bool(inter)
            top1 += r['pred'][0] in T
            exact += P == T
            p = len(inter) / len(P)
            rc = len(inter) / len(T)
            precs.append(p); recs_.append(rc)
            f1s.append(0.0 if p + rc == 0 else 2 * p * rc / (p + rc))
            jacs.append(len(inter) / len(P | T))
            tp_tot += len(inter); fp_tot += len(P - T); fn_tot += len(T - P)
        mp = tp_tot / (tp_tot + fp_tot) if tp_tot + fp_tot else 0.0
        mr = tp_tot / (tp_tot + fn_tot) if tp_tot + fn_tot else 0.0
        out[agent] = {
            'n valutabili': n,
            'hit': hit / n, 'top-1': top1 / n, 'exact set': exact / n,
            'P per esempio': statistics.mean(precs),
            'R per esempio': statistics.mean(recs_),
            'F1 per esempio': statistics.mean(f1s),
            'Jaccard': statistics.mean(jacs),
            'P micro': mp, 'R micro': mr,
            'F1 micro': 0.0 if mp + mr == 0 else 2 * mp * mr / (mp + mr),
        }
    return out


def show_stage2(mode):
    print(f'=== STADIO 2 — accuratezza CWE condizionata  (modalita\': {mode}) ===')
    hdr = ['n valutabili', 'hit', 'top-1', 'exact set', 'F1 per esempio',
           'P per esempio', 'R per esempio', 'Jaccard', 'F1 micro']
    for model in MODELS:
        st = stage2(DATA[model][mode])
        print(f'\n{model}')
        print('  ' + 'agente'.ljust(9) + ''.join(h.rjust(16) for h in hdr))
        for agent in AGENTS:
            s = st[agent]
            if not s.get('n valutabili'):
                print('  ' + agent.ljust(9) + '0'.rjust(16) + '  (nessuna istanza valutabile)')
                continue
            cells = []
            for h in hdr:
                v = s[h]
                cells.append((f'{v:.1%}' if isinstance(v, float) else str(v)).rjust(16))
            print('  ' + agent.ljust(9) + ''.join(cells))

show_stage2(PARSE_MODE)


In [ ]:
# F1 macro: calcolata per completezza, ma con ~1 esempio per classe non e' affidabile.
def macro_f1(rows):
    per = collections.defaultdict(lambda: [0, 0, 0])   # cwe -> [tp, fp, fn]
    for r in rows:
        if not (r['found'] and r['pred'] and r['truth']):
            continue
        P, T = set(r['pred']), set(r['truth'])
        for c in P & T: per[c][0] += 1
        for c in P - T: per[c][1] += 1
        for c in T - P: per[c][2] += 1
    f1s = []
    for tp, fp, fn in per.values():
        p = tp / (tp + fp) if tp + fp else 0.0
        r_ = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(0.0 if p + r_ == 0 else 2 * p * r_ / (p + r_))
    return (statistics.mean(f1s) if f1s else float('nan')), len(per)

print('=== F1 macro (da leggere con cautela) ===')
for model in MODELS:
    for agent in AGENTS:
        rows = [r for r in DATA[model][PARSE_MODE] if r['agent'] == agent]
        f1, k = macro_f1(rows)
        print(f'  {model:26s} {agent:8s} F1 macro {f1:6.1%}   su {k:3d} CWE distinte')
print()
print('AVVERTENZA: con 50 istanze e decine di CWE distinte quasi ogni classe compare')
print('una sola volta. La F1 macro e\' dominata dal rumore: riportarla solo se')
print('accompagnata dai conteggi, oppure ometterla.')


## Confronto tra modelli

Le due tabelle chiave affiancate. Riportare **sempre i conteggi assoluti** accanto
alle percentuali: con 50 istanze per agente, `6/27` dice la verità dove `22%`
suggerisce una precisione inesistente.


In [ ]:
print(f'=== SINTESI  (modalita\': {PARSE_MODE}) ===\n')
print('  ' + 'modello'.ljust(26) + 'agente'.ljust(9) +
      'concluso'.rjust(12) + 'recall'.rjust(12) + 'valutabili'.rjust(12) +
      'hit'.rjust(10) + 'top-1'.rjust(10) + 'F1 es.'.rjust(10))
for model in MODELS:
    s1, s2 = stage1(DATA[model][PARSE_MODE]), stage2(DATA[model][PARSE_MODE])
    for agent in AGENTS:
        a, b = s1[agent], s2[agent]
        n2 = b.get('n valutabili', 0)
        print('  ' + model.ljust(26) + agent.ljust(9) +
              f"{a['concluso']}/{a['n']}".rjust(12) +
              f"{a['recall condizionato']:.0%}".rjust(12) +
              str(n2).rjust(12) +
              (f"{b['hit']:.0%}".rjust(10) if n2 else '-'.rjust(10)) +
              (f"{b['top-1']:.0%}".rjust(10) if n2 else '-'.rjust(10)) +
              (f"{b['F1 per esempio']:.0%}".rjust(10) if n2 else '-'.rjust(10)))


In [ ]:
# Elenco delle istanze senza verdetto: sono i candidati per l'annotazione manuale
# (modalita' 'annotated'). Serve a capire quante conclusioni si stanno perdendo.
for model in MODELS:
    missing = [r for r in DATA[model]['tolerant'] if r['found'] is None]
    print(f'{model}: {len(missing)} triple senza verdetto leggibile')
    for r in missing:
        print(f"    {r['agent']:8s} {r['repo'][:40]:40s} ({r['source']})")
    print()


## Da fare

- **Corrispondenza per famiglia** (antenato CWE-1000): non implementata. Approcci,
  insidie e decisioni da dichiarare sono in `Literature/TODO_ANALISI_CWE.md`.
  Serve scaricare il catalogo da MITRE: la gerarchia **non** è in CVEfixes.
- **Annotazione manuale** delle risposte che concludono solo in prosa: creare
  `manual_verdicts.csv` e passare a `PARSE_MODE = 'annotated'`.
- **Verifica dei falsi positivi di agent3**: quando l'agente segnala una CWE diversa
  da quella attesa può aver trovato una vulnerabilità *diversa e reale*. I numeri di
  agent3 vanno quindi letti come **limite inferiore**.

## Cosa non scrivere in tesi

- Nessuna affermazione su accuratezza, precisione, falsi positivi o specificità del
  **rilevamento**: non ci sono negativi nel campione.
- Per agent1 e agent2 il verdetto sì/no è quasi sempre *sì* e quasi sempre corretto:
  non ha potere discriminante, l'unica metrica informativa è la CWE.
